In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2Model

torch.set_grad_enabled(False)

model = GPT2Model.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [2]:
#Embeddings

texts = ["Hello, good morning"]


def get_embeddings(input):
    wpe = model.wpe.weight
    wte = model.wte.weight
    encoded = tokenizer(input, return_tensors="pt")
    ids_input = encoded["input_ids"]
    num_tokens = ids_input.shape[1]
    positions = torch.arange(num_tokens)
    embeddings = wpe[positions] + wte[ids_input]

    return embeddings

embeddings = get_embeddings(texts)


In [4]:
def comparar(tensor_1, tensor_2):
    diferencias = (tensor_1 - tensor_2).abs()
    diferencia_max = diferencias.max().item()

    print(f"Diferencia máxima: {diferencia_max}")

    torch.testing.assert_close(tensor_1, tensor_2,atol=1e-4, rtol= 1e-5 )



In [5]:
#LayerNorm

def layernorm(tensor,weight,bias):
    mean = torch.mean(tensor, dim=-1, keepdim=True)
    eps = 1e-5
    variance = torch.var(tensor, dim=-1, keepdim=True, correction=0)

    output = ((tensor-mean)/torch.sqrt(variance+eps))* weight + bias
    return output


In [8]:
#MLP Implementation
import torch.nn.functional as fu

def mlp(tensor, weight_1, bias_1, weight_2, bias_2):
    inter = tensor @ weight_1 + bias_1
    inter_2 = fu.gelu(inter, approximate="tanh")
    output = inter_2 @ weight_2 + bias_2

    return output



In [12]:
#Implementing the attention layer

def attention(tensor, weight_1, bias_1, weight_2, bias_2):
    n_heads = 12
    head_dim = 768 // n_heads
    q,k,v = ((tensor @ weight_1) + bias_1).split(768, dim=-1)
    q = q.view(*q.shape[:-1],n_heads,head_dim)
    q = q.transpose(-3,-2)
    k = k.view(*k.shape[:-1],n_heads,head_dim)
    k = k.transpose(-3,-2)
    v = v.view(*v.shape[:-1],n_heads,head_dim)
    v = v.transpose(-3,-2)

    numerator = q @ k.transpose(-2,-1)
    denominator = head_dim ** 0.5
    mask = torch.tril(torch.ones(k[0].shape[-2],k[0].shape[-2], device=tensor.device))
    inter_mask = (numerator/denominator).masked_fill(mask==0, float("-inf"))
    softmax_res = torch.softmax(inter_mask, dim=-1)
    output_1 = softmax_res @ v

    output_1 = output_1.transpose(-3,-2)
    output_1 = output_1.reshape(*output_1.shape[:-2],n_heads*head_dim)

    output_2 = output_1@weight_2 + bias_2

    return output_2

In [14]:
#Full Block Implementation

def full_layer_block(tensor, w1, b1, w2, b2, w3, b3, w4, b4, w5, b5, w6, b6):

    inter_1 = layernorm(tensor,w1,b1)
    inter_1 = attention(inter_1,w2,b2,w3,b3)

    res_1 = tensor + inter_1

    inter_2 = layernorm(res_1,w4,b4)
    inter_2 = mlp(inter_2,w5,b5,w6,b6)

    res_2 = res_1 + inter_2

    return res_2


In [16]:
def gpt2_model(input_text):

    embeddings = get_embeddings(input_text)
    for layer in range (12):
        embeddings = full_layer_block(embeddings,model.h[layer].ln_1.weight,model.h[layer].ln_1.bias, model.h[layer].attn.c_attn.weight, model.h[layer].attn.c_attn.bias, model.h[layer].attn.c_proj.weight, model.h[layer].attn.c_proj.bias,model.h[layer].ln_2.weight, model.h[layer].ln_2.bias, model.h[layer].mlp.c_fc.weight, model.h[layer].mlp.c_fc.bias, model.h[layer].mlp.c_proj.weight, model.h[layer].mlp.c_proj.bias)

    embeddings = layernorm(embeddings, model.ln_f.weight, model.ln_f.bias)

    return embeddings


In [19]:
#Final test

input = ["Hello world how are you"]

wpe = model.wpe.weight
wte = model.wte.weight
encoded = tokenizer(input, return_tensors="pt")
ids_input = encoded["input_ids"]


mi_resultado_gpt = gpt2_model("Hello world how are you")

gpt_oficial = model(ids_input).last_hidden_state

comparar(mi_resultado_gpt,gpt_oficial)

Diferencia máxima: 5.340576171875e-05
